# Autoencoder on plausibilitree-DL point clouds

Bare-bones PointNet-style **autoencoder** (PyTorch), unsupervised — it only
reads the feature files from `prepare.py` (labels are not used):

- `features/<plot_uid>.npz` — `points` (N, 3) float32 plus every extra point
  dimension (`semantic_code`, `tree_id`, `shrub_id`, `classification`,
  `branch_code`, `branch_id`, `height_above_ground`) and a `meta` JSON string

One plot = one sample. Pick the device (CPU / Apple GPU via MPS / CUDA), the
feature directory, the extra point channels to train on (`EXTRA_DIMS`), and
how many plots to load (`N_SAMPLES`). Set `N_SAMPLES = 1` to load a single
sample and tweak the model until it overfits it.

Requires: `torch`, `numpy` (and `matplotlib` only for the optional plot).

In [ ]:
import glob
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# ---- device: CPU / Apple integrated GPU (MPS) / CUDA cores ---------------
DEVICE_CHOICE = "auto"  # "auto" | "cpu" | "mps" | "cuda"


def pick_device(choice: str) -> torch.device:
    if choice in ("cuda", "auto") and torch.cuda.is_available():
        return torch.device("cuda")
    if choice in ("mps", "auto") and torch.backends.mps.is_available():
        return torch.device("mps")  # Apple integrated GPU
    if choice != "auto":
        print(f"warning: {choice!r} not available, falling back to CPU")
    return torch.device("cpu")


device = pick_device(DEVICE_CHOICE)
print("device:", device)

In [ ]:
# ---- user config -----------------------------------------------------------
FEATURES_DIR = "../dl_dataset/features"  # prepare.py -o <out>/features
EXTRA_DIMS = ["semantic_code"]           # extra per-point channels to train on,
                                         # stacked onto xyz (input = 3 + len(EXTRA_DIMS))
N_POINTS = 1024    # each cloud is padded/tiled or truncated to this many points
N_SAMPLES = 1      # how many plots to load: an int, or "all"; 1 = overfit one sample
LATENT_DIM = 128   # bottleneck size
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
SEED = 42

_first = sorted(glob.glob(os.path.join(FEATURES_DIR, "*.npz")))
if not _first:
    raise FileNotFoundError(f"no feature files in {FEATURES_DIR!r} - run prepare.py first")
with np.load(_first[0]) as z:
    print("available feature arrays:", z.files)

In [ ]:
class CloudDataset(Dataset):
    """One plot per sample: a (C, N_POINTS) feature matrix from the .npz
    (points + EXTRA_DIMS channels). Optionally limited to the first
    n_samples plots."""

    def __init__(self, features_dir, extra_dims, n_points, n_samples=None):
        self.n_points = n_points
        self.extra_dims = list(extra_dims)
        feats = sorted(glob.glob(os.path.join(features_dir, "*.npz")))
        if n_samples is not None:
            feats = feats[:n_samples]
        if not feats:
            raise FileNotFoundError(f"no feature files in {features_dir!r}")
        self.samples = feats
        self.names = [os.path.splitext(os.path.basename(f))[0] for f in feats]
        with np.load(feats[0]) as z:
            available = set(z.files)
        missing = [k for k in self.extra_dims if k not in available]
        if missing:
            raise KeyError(f"feature dims {missing} not in the .npz; "
                           f"available: {sorted(available)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        with np.load(self.samples[i]) as z:
            cols = [z["points"].astype(np.float32)]          # (N, 3)
            for k in self.extra_dims:
                cols.append(z[k].astype(np.float32)[:, None])  # (N, 1)
        mat = np.concatenate(cols, axis=1)                   # (N, C)
        # fix the point count: tile if short, truncate if long
        n = len(mat)
        if n >= self.n_points:
            mat = mat[: self.n_points]
        else:
            reps = int(np.ceil(self.n_points / n))
            mat = np.tile(mat, (reps, 1))[: self.n_points]
        return torch.from_numpy(mat.T.copy())                 # (C, N_POINTS)


dataset = CloudDataset(FEATURES_DIR, EXTRA_DIMS, N_POINTS,
                       n_samples=None if N_SAMPLES == "all" else N_SAMPLES)
print(f"loaded {len(dataset)} sample(s): {dataset.names}")
print(f"input channels = {3 + len(EXTRA_DIMS)} (xyz + {EXTRA_DIMS})")

In [ ]:
class PointNetAE(nn.Module):
    """Bare-bones PointNet-style autoencoder: shared point MLP -> global max
    pool -> FC encoder to a latent vector -> FC decoder back to (C, N)."""

    def __init__(self, in_channels, n_points, latent_dim):
        super().__init__()
        self.in_channels = in_channels
        self.n_points = n_points
        self.enc = nn.Sequential(
            nn.Conv1d(in_channels, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(),
        )
        self.enc_fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, n_points * in_channels),
        )

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        h = torch.max(self.enc(x), dim=2).values  # (B, 1024)
        z = self.enc_fc(h)                        # (B, latent_dim)
        out = self.dec_fc(z).view(-1, self.in_channels, self.n_points)
        return out, z


model = PointNetAE(in_channels=3 + len(EXTRA_DIMS), n_points=N_POINTS,
                   latent_dim=LATENT_DIM).to(device)
print(model)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    n = 0
    for x in loader:
        x = x.to(device)
        if x.shape[0] == 1:
            # single-sample overfit mode: BatchNorm needs >1 row per batch
            x = x.repeat(BATCH_SIZE, 1, 1)
        opt.zero_grad()
        recon, _ = model(x)
        loss = criterion(recon, x)
        loss.backward()
        opt.step()
        total += loss.item() * x.shape[0]
        n += x.shape[0]
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(f"epoch {epoch:3d}/{EPOCHS}  recon MSE {total / max(n, 1):.6f}")

In [ ]:
# ---- reconstruct the first sample and report the error --------------------
model.eval()
with torch.no_grad():
    x = dataset[0][None].to(device)  # (1, C, N); eval mode so BatchNorm is fine at B=1
    recon, z = model(x)
    mse = criterion(recon, x).item()
print(f"sample {dataset.names[0]}: recon MSE {mse:.6f}, latent {tuple(z.shape)}")

try:
    import matplotlib.pyplot as plt
    o = x[0, :2].cpu().numpy()        # original points, x-y projection
    r = recon[0, :2].cpu().numpy()    # reconstructed points
    plt.scatter(o[0], o[1], s=4, label="original")
    plt.scatter(r[0], r[1], s=4, label="reconstructed")
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("y")
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping plot")